# GravTide3D: A Simple Example

**GravTide3D** calculates the gravitational potential and tidal field from a three-dimensional mass-density distribution.

In this example, we start from a 2D H$_2$ column-density map of OMC-1 and use the [Volume Density Mapper (VDM)](https://github.com/gxli/volume-density-mapper) to reconstruct the 3D density field.

**VDM is not required by GravTide3D.** If you already have a 3D mass-density cube from a simulation, tomography, or another reconstruction method, you can skip the VDM step and use that cube directly. For observational studies where only a 2D column-density map is available, VDM provides one way to estimate the required 3D density structure. Reconstruction choices and the periodic FFT boundary condition can affect the scientific interpretation. The original FITS dataset citation and redistribution terms still need maintainer confirmation (see `examples/README.md`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.wcs import WCS

from volume_density_mapper import density_reconstruction_3d
from gravtide3d import gravitational_potential, tidal_eigensystem

## 1. Example data

Load the OMC-1 H$_2$ column-density map and convert it to mass surface density.

The physical pixel size is calculated from the angular pixel size and the adopted source distance.

**Data credit:** This OMC-1 H₂ column-density cutout is derived from the
Herschel Gould Belt Survey (HGBS). Please cite André et al. (2010) and the
[HGBS dataset DOI (10.26131/IRSA72)](https://doi.org/10.26131/IRSA72).
The parent map and redistribution rights for this particular cutout still
need independent confirmation; see [examples/README.md](README.md).


In [ ]:
nh2 = fits.getdata("omc1_nh2.fits")
header = fits.getheader("omc1_nh2.fits")
wcs = WCS(header)

pc = 3.086e18
distance = 414.0  # pc
dx = abs(header["CDELT2"]) / 180 * np.pi * distance * pc

# N(H2) [cm^-2] -> mass surface density [g cm^-2]
# mu_H2 = mean mass per H2 molecule in units of m_H.
m_H = 1.6735575e-24  # g, mass of hydrogen atom
mu_H2 = 2.8
mh2 = mu_H2 * m_H
surface_density = nh2 * mh2
log_nh2 = np.log10(np.where(nh2 > 0, nh2, np.nan))

plt.figure(figsize=(6, 5))
ax = plt.subplot(projection=wcs)
im = ax.imshow(log_nh2, origin="lower", cmap="magma")
plt.colorbar(im, ax=ax, label=r"log Column Density [cm$^{-2}$]")
ax.contour(log_nh2, colors="w", levels=np.arange(22, 24, 0.5))
ax.set_title("OMC-1 Column Density")
plt.show()

## 2. Reconstruct the 3D density field

For this observational example, we use **VDM** to reconstruct a 3D mass-density cube from the 2D column-density map.

GravTide3D itself only needs:

```python
density3d      # 3D mass density [g cm^-3], shape (Nz, Ny, Nx)
voxel_size     # physical voxel size (dz, dy, dx) [cm]
```

If you already have a 3D density cube, you can skip this section and continue directly to **Gravitational Potential** or **Tidal Field**.

In [ ]:
density3d = density_reconstruction_3d(surface_density, dx)

# GravTide3D uses voxel order (dz, dy, dx)
voxel_size = (dx, dx, dx)

# Density-weighted LOS mean density
density_sum_los = np.sum(density3d, axis=0)
nH2d = np.divide(
    np.sum(density3d**2, axis=0),
    density_sum_los,
    out=np.full_like(density_sum_los, np.nan, dtype=float),
    where=density_sum_los > 0,
)

plt.figure(figsize=(6, 5))
ax = plt.subplot(projection=wcs)
im = ax.imshow(np.log10(nH2d / mh2), origin="lower", cmap="magma")
plt.colorbar(im, ax=ax, label=r"log Volume Density [cm$^{-3}$]")
ax.contour(log_nh2, colors="w", levels=np.arange(22, 24, 0.5))
ax.set_title("Volume Density")
plt.show()

print("3D density shape:", density3d.shape)

## 3. Gravitational potential

The gravitational potential is calculated directly from the 3D mass-density cube and its physical voxel size.

**CPU is used by default.** To use a GPU, add `backend="gpu"` (and optionally choose a device with `devices=0`).

In [ ]:
potential = gravitational_potential(
    density3d,
    voxel_size,
)

phi = potential.phi

# Density-weighted projection along the line of sight
phi_weighted = np.divide(
    np.sum(phi * density3d, axis=0),
    density_sum_los,
    out=np.full_like(density_sum_los, np.nan, dtype=float),
    where=density_sum_los > 0,
)

plt.figure(figsize=(6, 5))
ax = plt.subplot(projection=wcs)
im = ax.imshow(phi_weighted, origin="lower", cmap="viridis")
plt.colorbar(im, ax=ax, label=r"Density-weighted $\Phi$ [cm$^2$ s$^{-2}$]")
ax.contour(log_nh2, colors="w", levels=np.arange(22, 24, 0.5))
ax.set_title("Gravitational Potential")
plt.show()

## 4. Tidal field

The tidal eigensystem can be calculated directly from the same 3D density cube.

The returned eigenvalues are ordered as $\lambda_{\min}$, $\lambda_{\rm mid}$, and $\lambda_{\max}$. By default, the eigenvalues already include the constant DC correction.

**CPU is used by default.** The final section shows the optional GPU version. Both use the same tidal sign and DC conventions. The Fourier solver has periodic boundaries and does not zero-pad the observational cube.

In [ ]:
tidal = tidal_eigensystem(
    density3d,
    voxel_size,
)

eigenvalues = tidal.eigenvalues
eigenvectors = tidal.eigenvectors

lambda_min = eigenvalues[..., 0]
zmid = lambda_min.shape[0] // 2

plt.figure(figsize=(6, 5))
ax = plt.subplot(projection=wcs)
im = ax.imshow(
    lambda_min[zmid], origin="lower", cmap="RdBu_r",
    vmin=-1e-24, vmax=1e-24,
)
plt.colorbar(im, ax=ax, label=r"$\lambda_{\min}$ [s$^{-2}$]")
ax.contour(log_nh2, colors="k", levels=np.arange(22, 24, 0.5))
ax.set_title(r"Tidal Field — $\lambda_{\min}$")
plt.show()

## 5. GPU acceleration (optional)

This section requires a CUDA-capable GPU and a compatible **CuPy** installation.
Everything above runs on the CPU alone. Select `backend="gpu"` and
`devices=0` for the first GPU; in-core GPU results are CuPy arrays, so
transfer the displayed slice to the CPU with `.get()` before plotting.

For a direct visual comparison, the GPU and CPU figures use the same fixed
color scale. A quantitative CPU/GPU eigenvalue regression result and remaining
GPU validation limits are documented in `../DEVELOPMENT_NOTES.md`.


In [ ]:
# Optional — tidal eigensystem on one GPU. Skips cleanly without CuPy.
try:
    import cupy as cp
except ImportError:
    cp = None

if cp is None:
    print("GPU example skipped: install the CuPy wheel for your CUDA version.")
elif cp.cuda.runtime.getDeviceCount() < 1:
    print("GPU example skipped: no CUDA GPU detected.")
else:
    tidal_gpu = tidal_eigensystem(
        density3d,
        voxel_size,
        backend="gpu",
        devices=0,
        memory="in_core",
    )

    # GPU outputs are CuPy arrays; transfer only the plotted eigenvalue field.
    lambda_min_gpu = tidal_gpu.eigenvalues[..., 0].get()
    zmid = lambda_min_gpu.shape[0] // 2

    plt.figure(figsize=(6, 5))
    ax = plt.subplot(projection=wcs)
    im = ax.imshow(
        lambda_min_gpu[zmid], origin="lower", cmap="RdBu_r",
        vmin=-1e-24, vmax=1e-24,
    )
    plt.colorbar(im, ax=ax, label=r"$\lambda_{\min}$ [s$^{-2}$]")
    ax.contour(log_nh2, colors="k", levels=np.arange(22, 24, 0.5))
    ax.set_title(r"GPU Tidal Field — $\lambda_{\min}$")
    plt.show()
